In [18]:
!pip install torch torchvision --quiet
!pip install einops --quiet
!pip install transformers --quiet

In [19]:
import sys
import os
import glob
import cv2
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    jaccard_score,
    classification_report
)

from pathlib import Path

import os
import re
import time
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from einops import rearrange


In [20]:
dataset_dir = Path('../EWS-Dataset')

train_dir = dataset_dir / 'train'
test_dir = dataset_dir / 'test'
validation_dir = dataset_dir / 'validation'

In [21]:
PIXELS_PER_IMAGE = 5000

FEATURE_MODES = {
    "RGB": ["r", "g", "b"],
    "RGB+HSV": ["r", "g", "b", "h", "s", "v"],
    "RGB+HSV+Texture": ["r", "g", "b", "h", "s", "v", "local_mean", "local_var"],
    "RGB+HSV+Texture+VegIdx": ["r", "g", "b", "h", "s", "v", "local_mean", "local_var", "exg", "ndi", "veg", "exg_blur3", "exg_blur7"],
}

In [28]:
class WheatDataset(Dataset):
    def __init__(self, img_dir, transform=None, mask_transform=None):
        self.img_dir = Path(img_dir)
        self.images = sorted([
            p for p in self.img_dir.glob('*.png')
            if '_mask' not in p.stem
        ])
        self.transform = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        mask_path = self.img_dir / (img_path.stem + '_mask.png')

        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')

        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        mask = (mask < 0.5).float()
        return image, mask
        
img_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

mask_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

train_dataset = WheatDataset('../EWS-Dataset/train',      img_transform, mask_transform)
val_dataset   = WheatDataset('../EWS-Dataset/validation', img_transform, mask_transform)
test_dataset  = WheatDataset('../EWS-Dataset/test',       img_transform, mask_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8)
test_loader  = DataLoader(test_dataset,  batch_size=8)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
img, mask = train_dataset[0]
print(f"Image shape: {img.shape}")
print(f"Mask shape:  {mask.shape}")
print(f"Mask unique values: {mask.unique()}")

Train: 142 | Val: 24 | Test: 24
Image shape: torch.Size([3, 224, 224])
Mask shape:  torch.Size([1, 224, 224])
Mask unique values: tensor([0., 1.])


In [23]:
import torch.nn.functional as F

def dice_loss(pred, target, smooth=1):
    pred = pred.contiguous().view(-1)
    target = target.contiguous().view(-1)
    intersection = (pred * target).sum()
    return 1 - (2 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

def combined_loss(logits, masks):
    logits = F.interpolate(logits, size=masks.shape[-2:],
                           mode = 'bilinear', align_corners=False)
    ce = F.cross_entropy(logits, masks)
    pred = logits.softmax(dim=1)[:,1]
    dl = dice_loss(pred, masks.float())
    return ce + dl

In [24]:
from transformers import SegformerForSemanticSegmentation

model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b0",
    num_labels=2,                  # 0=background, 1=plant
    ignore_mismatched_sizes=True
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f"Running on: {device}")

You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/mit-b0
Key                                           | Status     | 
----------------------------------------------+------------+-
classifier.weight                             | UNEXPECTED | 
classifier.bias                               | UNEXPECTED | 
decode_head.classifier.weight                 | MISSING    | 
decode_head.batch_norm.num_batches_tracked    | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.bias   | MISSING    | 
decode_head.batch_norm.weight                 | MISSING    | 
decode_head.classifier.bias                   | MISSING    | 
decode_head.linear_c.{0, 1, 2, 3}.proj.weight | MISSING    | 
decode_head.linear_fuse.weight                | MISSING    | 
decode_head.batch_norm.running_mean           | MISSING    | 
decode_head.batch_norm.bias                   | MISSING    | 
decode_head.batch_norm.running_var            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different ta

Running on: cpu


In [29]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np

optimizer = AdamW(model.parameters(), lr=6e-5, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=30)

def compute_iou(pred_mask, true_mask):
    pred_mask = pred_mask.bool()
    true_mask = true_mask.bool()
    intersection = (pred_mask & true_mask).sum().float()
    union = (pred_mask | true_mask).sum().float()

    return (intersection / (union + 1e-6)).item()

best_val_iou = 0.0

for epoch in range(30):
    # --- Train ---
    model.train()
    train_losses = []
    for images, masks in train_loader:
        images = images.to(device)
        masks  = masks.to(device).long().squeeze(1)   # (B, H, W)

        outputs = model(pixel_values=images)
        loss    = combined_loss(outputs.logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    scheduler.step()

    # --- Validate ---
    model.eval()
    val_ious = []
    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks  = masks.to(device).long().squeeze(1)

            outputs = model(pixel_values=images)
            logits  = F.interpolate(outputs.logits, size=masks.shape[-2:],
                                    mode='bilinear', align_corners=False)
            preds = logits.argmax(dim=1)
            for p, m in zip(preds, masks):
                val_ious.append(compute_iou(p, m))

    mean_iou = np.mean(val_ious)
    print(f"Epoch {epoch+1:02d} | Loss: {np.mean(train_losses):.4f} | Val IoU: {mean_iou:.4f}")

    # Save best model
    if mean_iou > best_val_iou:
        best_val_iou = mean_iou
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"Saved best model (IoU: {mean_iou:.4f})")

Epoch 01 | Loss: 1.2182 | Val IoU: 0.3861
Saved best model (IoU: 0.3861)
Epoch 02 | Loss: 1.0244 | Val IoU: 0.4753
Saved best model (IoU: 0.4753)
Epoch 03 | Loss: 0.8809 | Val IoU: 0.5260
Saved best model (IoU: 0.5260)
Epoch 04 | Loss: 0.7612 | Val IoU: 0.5852
Saved best model (IoU: 0.5852)
Epoch 05 | Loss: 0.6821 | Val IoU: 0.6081
Saved best model (IoU: 0.6081)
Epoch 06 | Loss: 0.6197 | Val IoU: 0.6185
Saved best model (IoU: 0.6185)
Epoch 07 | Loss: 0.5989 | Val IoU: 0.6277
Saved best model (IoU: 0.6277)
Epoch 08 | Loss: 0.5609 | Val IoU: 0.6281
Saved best model (IoU: 0.6281)
Epoch 09 | Loss: 0.5480 | Val IoU: 0.6285
Saved best model (IoU: 0.6285)
Epoch 10 | Loss: 0.5479 | Val IoU: 0.6434
Saved best model (IoU: 0.6434)
Epoch 11 | Loss: 0.5275 | Val IoU: 0.6378
Epoch 12 | Loss: 0.5094 | Val IoU: 0.6405
Epoch 13 | Loss: 0.4957 | Val IoU: 0.6445
Saved best model (IoU: 0.6445)
Epoch 14 | Loss: 0.4701 | Val IoU: 0.6469
Saved best model (IoU: 0.6469)
Epoch 15 | Loss: 0.4705 | Val IoU: 0.651

In [31]:
print(mean_iou)

0.6482674467066923


In [ ]:
all_preds = []
all_masks = []

model.eval()
with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device).long().squeeze(1)

SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 32, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 160, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((160,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(160, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  